# 🎭 **Masked Language Modeling with BERT**

Explore masked language modeling (MLM) using the BERT model to understand context and predict missing words in sentences.

## 🛠️ Setup and Installation

Begin by installing the necessary libraries to manage data processing and modeling.

In [1]:
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 52.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.47.1
    Uninstalling transformers-4.47.1:
      Successfully uninstalled transformers-4.47.1


## 📚 Importing Libraries

Import essential modules for our tasks.

In [2]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
import pandas as pd
import numpy as np
from scipy.special import softmax

## 🤖 Model Setup

Load the pre-trained BERT model and tokenizer, specifically designed for masked language modeling.

In [3]:
model_name = "bert-base-cased"

# Loading the pre-trained model and tokenizer
model = AutoModelForMaskedLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

## 🎭 Defining the Mask Token

Identify the mask token used by BERT to signify where predictions are needed in the sentence.

## ✏️ Creating the Input Sentence

Craft a sentence with a missing word indicated by the mask token, to test the model's predictive power.

In [4]:
# Defining the mask token
mask = tokenizer.mask_token

# Defining the sentence
sentence = f"I want to {mask} pizza for tonight."

# Tokenizing the sentence
tokens = tokenizer.tokenize(sentence)

In [44]:
sentence

'I want to [MASK] pizza for tonight.'

In [6]:
mask, tokens

('[MASK]', ['I', 'want', 'to', '[MASK]', 'pizza', 'for', 'tonight', '.'])

In [47]:
len(tokens)

8

## 🔍 Tokenization and Encoding

Tokenize and encode the sentence to format it properly for the model.

## 📈 Model Prediction

Feed the encoded inputs to the model and extract logits for predictions.

In [7]:
# Encoding the input sentence and getting model predictions
encoded_inputs = tokenizer(sentence, return_tensors="pt")
output = model(**encoded_inputs)

# Detaching the logits from the model output and converting to numpy array
logits = output.logits.detach().numpy()[0]

In [11]:
output.logits.shape

torch.Size([1, 10, 28996])

In [16]:
output.logits.detach()

tensor([[[ -7.3723,  -7.2489,  -7.4421,  ...,  -6.3119,  -5.9369,  -6.4257],
         [ -7.9311,  -8.2282,  -8.0326,  ...,  -6.7387,  -6.4877,  -6.9525],
         [-12.0500, -11.7972, -12.5776,  ...,  -8.4518,  -6.7310,  -8.2586],
         ...,
         [-10.2204, -10.4315,  -9.9993,  ...,  -7.9570,  -6.7194,  -9.3618],
         [-12.4471, -12.5367, -12.5614,  ...,  -9.9086,  -9.4219, -11.1770],
         [-14.3657, -14.5227, -15.0017,  ..., -11.9715, -11.6569, -13.4498]]])

In [15]:
output.logits.detach().numpy()

array([[[ -7.3722925,  -7.2488613,  -7.4421444, ...,  -6.311862 ,
          -5.936892 ,  -6.425681 ],
        [ -7.9311185,  -8.2282095,  -8.032589 , ...,  -6.7387457,
          -6.4877234,  -6.9525247],
        [-12.050008 , -11.797209 , -12.577608 , ...,  -8.451776 ,
          -6.7310185,  -8.258566 ],
        ...,
        [-10.22041  , -10.4314785,  -9.999257 , ...,  -7.9569917,
          -6.7193975,  -9.361793 ],
        [-12.447125 , -12.536707 , -12.561406 , ...,  -9.908555 ,
          -9.421911 , -11.176952 ],
        [-14.365711 , -14.522715 , -15.001671 , ..., -11.971546 ,
         -11.65692  , -13.449785 ]]], dtype=float32)


## 🔎 Analyzing Predictions

Retrieve logits for the masked token and calculate confidence scores for possible replacements.

In [18]:
logits.shape

(10, 28996)

In [19]:
tokens.index(mask)

3

In [22]:
logits[tokens.index(mask) + 1]

array([-6.714628 , -6.379109 , -6.1184893, ..., -5.651309 , -3.6572778,
       -4.9947314], dtype=float32)

In [23]:
# Extracting the logits for the masked token and calculating the confidence scores
masked_logits = logits[tokens.index(mask) + 1]
confidence_scores = softmax(masked_logits)

In [26]:
confidence_scores

array([2.9159888e-10, 4.0784978e-10, 5.2928079e-10, ..., 8.4446000e-10,
       6.2026344e-09, 1.6282734e-09], dtype=float32)

In [27]:
confidence_scores.sum()

1.0

## 📝 Displaying Top Predictions

Cycle through the top 5 predicted tokens, substituting the masked token in the original sentence to show the model's suggestions.


In [34]:
np.argsort(confidence_scores)

array([15318, 18715, 18487, ...,  3940,  1243,  1138])

In [37]:
np.argsort(confidence_scores)[::-1]

array([ 1138,  1243,  3940, ..., 18487, 18715, 15318])

In [38]:
confidence_scores[np.argsort(confidence_scores)[::-1]]

array([2.5729063e-01, 1.7849593e-01, 1.5555556e-01, ..., 5.3032790e-13,
       3.8729894e-13, 2.1802671e-13], dtype=float32)

In [39]:
confidence_scores[np.argsort(confidence_scores)[::-1][:5]]

array([0.25729063, 0.17849593, 0.15555556, 0.11422409, 0.0982304 ],
      dtype=float32)

In [49]:
# Iterating over the top 5 predicted tokens and printing the sentences with the masked token replaced
for token_id in np.argsort(confidence_scores)[::-1][:5]:
    pred_token = tokenizer.decode(token_id)
    score = confidence_scores[token_id]
    print('Token ID: {}, Pred Token: {}, Confidence score: {}'.format(token_id, pred_token, score))

    # print(pred_token, score)
    print(sentence.replace(mask, pred_token))

Token ID: 1138, Pred Token: have, Confidence score: 0.25729063153266907
I want to have pizza for tonight.
Token ID: 1243, Pred Token: get, Confidence score: 0.17849592864513397
I want to get pizza for tonight.
Token ID: 3940, Pred Token: eat, Confidence score: 0.15555556118488312
I want to eat pizza for tonight.
Token ID: 1294, Pred Token: make, Confidence score: 0.11422409117221832
I want to make pizza for tonight.
Token ID: 1546, Pred Token: order, Confidence score: 0.09823039919137955
I want to order pizza for tonight.
